In [33]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="📊 CSV Column Analyzer", layout="wide")

st.title("📊 CSV Column Analyzer with Editable Types + Unique Value Analysis")

# ========================
# Upload CSV
# ========================
uploaded_file = st.file_uploader("Upload your CSV file", type=["csv"])

if uploaded_file:
    df = pd.read_csv(uploaded_file)
    st.success(f"✅ Loaded `{uploaded_file.name}` with {df.shape[0]} rows & {df.shape[1]} columns")

    st.subheader("🔹 Preview of Data")
    st.dataframe(df.head())

    # ========================
    # Step 1: Column Summary
    # ========================
    summary = []
    for col in df.columns:
        dtype = df[col].dtype
        unique_vals = df[col].nunique()
        total_vals = len(df[col])

        # Detect simple type
        if unique_vals == total_vals:
            detected_type = "Unique Identifier"
        elif np.issubdtype(dtype, np.number):
            detected_type = "Number"
        else:
            detected_type = "String"

        # Default suggestion
        if detected_type == "Unique Identifier":
            suggested = "Drop"
        elif detected_type == "Number":
            suggested = "Continuous"
        else:
            suggested = "Nominal"

        summary.append([col, dtype, unique_vals, detected_type, suggested])

    summary_df = pd.DataFrame(summary, columns=["Column", "Dtype", "Unique Values", "Detected Type", "Suggested Type"])

    st.subheader("📌 Column Summary (Choose Suggested Type)")
    edited_df = st.data_editor(
        summary_df,
        use_container_width=True,
        num_rows="fixed",
        key="editable_summary",
        column_config={
            "Suggested Type": st.column_config.SelectboxColumn(
                "Suggested Type",
                help="Select how you want to treat this column",
                options=["Drop", "Continuous", "Discrete", "Ordinal", "Nominal"],
                required=True
            )
        }
    )

    # ========================
    # Step 2: Apply Changes
    # ========================
    if st.button("✅ Apply Changes"):
        processed_df = df.copy()
        changes = {}

        for _, row in edited_df.iterrows():
            col, suggestion = row["Column"], row["Suggested Type"]
            if suggestion == "Drop":
                processed_df.drop(columns=[col], inplace=True)
                changes[col] = "Dropped"
            elif suggestion in ["Continuous", "Discrete"]:
                processed_df[col] = pd.to_numeric(processed_df[col], errors="coerce")
                changes[col] = f"Converted to {suggestion}"
            elif suggestion in ["Ordinal", "Nominal"]:
                processed_df[col] = processed_df[col].astype(str)
                changes[col] = f"Converted to {suggestion}"

        st.success("✅ Changes applied successfully!")
        st.dataframe(processed_df.head())

        changes_df = pd.DataFrame(list(changes.items()), columns=["Column", "Action"])
        st.subheader("🔄 User Changes Summary (Confirm before proceeding)")
        st.dataframe(changes_df)

        # Download updated CSV
        csv = processed_df.to_csv(index=False).encode("utf-8")
        st.download_button("📥 Download Updated CSV", data=csv, file_name="processed.csv", mime="text/csv")

        # ========================
        # Step 3: Unique Value Analysis
        # ========================
        st.subheader("🔍 Unique Value Analysis")
        num_cols = processed_df.select_dtypes(include=np.number).columns.tolist()
        if num_cols:
            col_choice = st.selectbox("Select Numerical Column for Analysis", num_cols)
            group_col = st.selectbox("Select Column to Group By", processed_df.columns)

            if group_col and col_choice:
                if processed_df[group_col].nunique() == len(processed_df):
                    st.warning(f"⚠️ The column '{group_col}' looks like a unique identifier. Grouping by it may not be useful.")
                else:
                    agg_df = processed_df.groupby(group_col)[col_choice].sum().reset_index()
                    total = agg_df[col_choice].sum()
                    agg_df[f"{col_choice}_%"] = (agg_df[col_choice] / total * 100).round(2)

                    st.dataframe(agg_df)

        # ========================
        # Step 4: Multi-Column Unique Value Analysis
        # ========================
        st.subheader("📊 Multi-Column Unique Value Analysis")

        # Only allow valid group columns (not unique IDs)
        valid_groupby_cols = [
            col for col in processed_df.columns
            if processed_df[col].nunique() < 0.5 * len(processed_df)
        ]

        group_col_multi = st.selectbox("Select Column to Group By", valid_groupby_cols)
        num_cols_multi = st.multiselect("Select Numerical Columns for Analysis", num_cols)

        if group_col_multi and num_cols_multi:
            if processed_df[group_col_multi].nunique() == len(processed_df):
                st.warning(f"⚠️ The column '{group_col_multi}' looks like a unique identifier. Grouping is not meaningful.")
            else:
                agg_df_multi = processed_df.groupby(group_col_multi)[num_cols_multi].sum().reset_index()
                for col in num_cols_multi:
                    total = agg_df_multi[col].sum()
                    agg_df_multi[f"{col}_%"] = (agg_df_multi[col] / total * 100).round(2)

                st.dataframe(agg_df_multi)


2025-08-18 22:45:49.371 
  command:

    streamlit run C:\ProgramData\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]


In [ ]:
!streamlit run web.py